# Majority-class baseline

This notebook reads the fixed source-image split from `01_data_preparation.ipynb`, predicts the training-set majority class for every validation pixel, and reports the segmentation metrics.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
from PIL import Image
import tensorflow as tf

SEED = 42
PATCH_SIZE = 256
BATCH_SIZE = 8
IGNORE_LABEL = 255
NUM_CLASSES = 3
CLASS_COLORS = np.array([[199, 199, 199], [31, 119, 180], [255, 127, 14]], dtype=np.uint8)
CLASS_NAMES = ['background/soil', 'crop/sorghum', 'weeds']
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'project_plan.md').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final-project'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
split_path = ARTIFACT_DIR / 'split.json'
if not split_path.exists():
    raise FileNotFoundError('Run 01_data_preparation.ipynb first.')
split_record = json.loads(split_path.read_text(encoding='utf-8'))
train_images = [PROJECT_ROOT / path for path in split_record['train_images']]
val_images = [PROJECT_ROOT / path for path in split_record['validation_images']]
def mask_for(path):
    return path.parent.parent / 'msk' / f'{path.stem}.png'
train_masks = [mask_for(path) for path in train_images]
val_masks = [mask_for(path) for path in val_images]
print('Loaded split:', split_path)
print('Training sources:', len(train_images), 'Validation sources:', len(val_images))


In [ ]:
def generate_patches(image_paths, mask_paths):
    for image_path, mask_path in zip(image_paths, mask_paths):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            rgb = np.asarray(image.convert('RGB'))
            mask_rgb = np.asarray(mask.convert('RGB'))
        labels = np.full(mask_rgb.shape[:2], IGNORE_LABEL, dtype=np.uint8)
        for class_id, color in enumerate(CLASS_COLORS):
            labels[np.all(mask_rgb == color, axis=-1)] = class_id
        for y in range(0, labels.shape[0], PATCH_SIZE):
            for x in range(0, labels.shape[1], PATCH_SIZE):
                height = min(PATCH_SIZE, labels.shape[0] - y)
                width = min(PATCH_SIZE, labels.shape[1] - x)
                label_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.int32)
                weight_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
                label_patch[:height, :width] = labels[y:y + height, x:x + width]
                weight_patch[:height, :width] = 1.0
                yield label_patch, weight_patch

def make_label_dataset(image_paths, mask_paths):
    dataset = tf.data.Dataset.from_generator(
        lambda: generate_patches(image_paths, mask_paths),
        output_signature=(tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.int32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.float32)),
    )
    return dataset.batch(BATCH_SIZE)

def count_patches(image_paths):
    total = 0
    for path in image_paths:
        with Image.open(path) as image:
            total += math.ceil(image.height / PATCH_SIZE) * math.ceil(image.width / PATCH_SIZE)
    return total

train_labels = make_label_dataset(train_images, train_masks)
val_labels = make_label_dataset(val_images, val_masks)
train_steps = math.ceil(count_patches(train_images) / BATCH_SIZE)
val_steps = math.ceil(count_patches(val_images) / BATCH_SIZE)
print('Training steps:', train_steps, 'Validation steps:', val_steps)


In [ ]:
class_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for labels, weights in train_labels.take(train_steps):
    labels_np, weights_np = labels.numpy(), weights.numpy()
    for class_id in range(NUM_CLASSES):
        class_pixel_counts[class_id] += int(np.sum((labels_np == class_id) & (weights_np > 0)))
majority_class = int(np.argmax(class_pixel_counts))
print('Training pixel counts:', class_pixel_counts.tolist())
print('Majority class:', majority_class, CLASS_NAMES[majority_class])

def update_confusion(confusion, labels, predictions, weights):
    valid = tf.reshape(weights > 0, [-1])
    return confusion + tf.math.confusion_matrix(
        tf.boolean_mask(tf.reshape(labels, [-1]), valid),
        tf.boolean_mask(tf.reshape(predictions, [-1]), valid),
        num_classes=NUM_CLASSES, dtype=tf.float64,
    )

confusion = tf.zeros((NUM_CLASSES, NUM_CLASSES), dtype=tf.float64)
for labels, weights in val_labels.take(val_steps):
    predictions = tf.fill(tf.shape(labels), tf.cast(majority_class, tf.int32))
    confusion = update_confusion(confusion, labels, predictions, weights)
true_positives = tf.linalg.diag_part(confusion)
actual = tf.reduce_sum(confusion, axis=1)
predicted = tf.reduce_sum(confusion, axis=0)
iou = tf.math.divide_no_nan(true_positives, actual + predicted - true_positives).numpy()
dice = tf.math.divide_no_nan(2.0 * true_positives, actual + predicted).numpy()
scores = {'iou': iou.tolist(), 'dice': dice.tolist(), 'mean_iou': float(iou.mean()), 'weed_iou': float(iou[2])}
print('Confusion matrix:\n', confusion.numpy().astype(np.int64))
for class_id, name in enumerate(CLASS_NAMES):
    print(f'{name}: IoU={iou[class_id]:.4f}, Dice={dice[class_id]:.4f}')
print(f'Mean IoU: {scores["mean_iou"]:.4f}')
print(f'Weed IoU: {scores["weed_iou"]:.4f}')
(ARTIFACT_DIR / 'MAJORITY_BASELINE_SCORES.json').write_text(json.dumps(scores, indent=2) + '\n', encoding='utf-8')
